# Smart Microscopy v3

Adaptive overview -> target acquisition at two objectives, with
on-the-fly per-tile cellpose segmentation via the smart-analysis engine.

**Z handling:** z-wide everywhere; z-galvo never commanded (set it to
0 in LAS X before running for best accuracy -- preflight will warn).

All logic lives in `notebooks/workflow/`. Each cell here is a thin
wrapper. See `TARGET_ACQUISITION_DESIGN.md` for the full design.

Steps:

- **Cell 2** -- Config + LAS X connect (operator inputs).
- **Cell 3** -- Step 0: preflight (calibration, engine boot, register pipeline, telemetry).
- **(later)** -- Step 1 prepare_template, Step 2 read_scan_field, Step 3 build_focus_map,
  Step 4 run_overview_with_picks, Step 5 acquire_targets, Step 6 outputs + cleanup.
- **Last cell** -- always-run cleanup: `ctx.shutdown()` (idempotent).

In [1]:
import sys
from pathlib import Path

# Locate notebooks/workflow/ regardless of where Jupyter was launched.
# Covered launch modes:
#   - smart-microscopy/                            (repo root)
#   - .../navigator_expert/                        (parent of notebooks/)
#   - .../navigator_expert/notebooks/              (the notebooks dir itself)
#   - .../navigator_expert/<sibling-of-notebooks>/ (e.g., examples/, test/)
_HERE = Path.cwd()
for _c in [_HERE / "controller/vendor/leica/navigator_expert/notebooks",
           _HERE,
           _HERE / "notebooks",
           _HERE.parent / "notebooks"]:
    if (_c / "workflow" / "__init__.py").exists():
        sys.path.insert(0, str(_c.resolve()))
        break
else:
    raise RuntimeError(
        "Could not find notebooks/workflow/. Launch Jupyter from the "
        "smart-microscopy repo root, navigator_expert/, or "
        "navigator_expert/notebooks/."
    )

from LasxApi import PYLICamApiConnector as lasxApi
from workflow import Config

# --- LAS X connect (the notebook owns this; preflight receives an open client) ---
client = lasxApi.LasxApiClientPyModel
client.Connect("PythonClient")

# --- Operator inputs ---
# Stage XY limits come from boundary point markers placed in
# Navigator Expert (preferred); the physical envelope is read from
# stage.json. Z-wide limits come from stage.json. No coordinates
# typed by the operator here.
cfg = Config(
    # Slots and jobs
    source_slot=1,
    target_slot=2,
    acquisition_job="Overview",
    target_job="Target",
    af_job="AF Job",

    # Pick policy (top-N per overview tile)
    n_picks_per_tile=5,

    # Paths
    analysis_repo=Path(r"Z:\zmbstaff\10374\Protocols_Notes\thom\notes\repositories\smart-analysis"),
    output_root=Path(r"Z:\zmbstaff\10374\Protocols_Notes\thom\notes\repositories\smart-microscopy\controller\vendor\leica\navigator_expert\output\target_acquisition"),

    # Defaults made explicit
    feature="area",
    fov_bbox_margin=1.5,
    settle_after_objective_switch_s=3.0,
    smoke_test_pipeline=False,
)

print(f"Connected. Config built. Source slot {cfg.source_slot} -> target slot {cfg.target_slot}.")

Connected. Config built. Source slot 1 -> target slot 2.


In [2]:
from workflow import preflight

ctx = preflight(cfg, client)

C:\Users\local_t.de2\Temp\ipykernel_21600\1725752324.py:3: UserWarning: Conda env 'SMART--target_acquisition--main' not found under any of: ['C:\\ProgramData\\MinicondaZMB\\envs\\SMART--target_acquisition--main', 'C:\\ProgramData\\MinicondaZMB\\envs\\SMART--target_acquisition--main']. v0 stubs do not need it; create it before wiring real cellpose into segment_tile.
  ctx = preflight(cfg, client)


RuntimeError: drv.find_scanning_templates_dir() returned None. LAS X may not be installed/configured for this Windows user, or the kernel was launched without inheriting APPDATA. See navigator_expert.driver.find_scanning_templates_dir for the exact lookup logic.

## Step 1 -- Stage limits + template

Optionally place **point markers** in Navigator Expert at the corners of your
sample. The cell will derive XY stage limits from those markers (with
`cfg.limit_margin_um` padding), clamped to the physical envelope from
`stage.json` (any clamp is reported per axis).

If no markers are placed, the physical envelope is applied immediately
so any intermediate move stays bounded; XY narrowing is then deferred
to Step 2 (derived from the scan field, also clamped to the envelope).

The cell also strips the template and forces every job in the LRP to **z-wide**.

In [3]:
from workflow import prepare_template

prepare_template(ctx)

NameError: name 'ctx' is not defined

## Step 2 -- Scan field

Draw the scanning field in **Navigator Expert**, then run the next two cells.

In [ ]:
from workflow import read_scan_field, plot_scan_field

read_scan_field(ctx)
plot_scan_field(ctx)

## Step 3 -- Focus map

Place **focus markers** on the scan field in **Navigator Expert**, then run this cell.
Runs the AF job at each marker, reads back z-wide, fits a plane.

In [ ]:
from workflow import build_focus_map

focus_map = build_focus_map(ctx)
focus_map.plot(ctx)